# Qdrant Retrieval Evaluation Notebook

Этот ноутбук оценивает retrieval-качество для уже загруженных экспериментов в Qdrant.

Что делает:
1. Загружает query-векторы (`record_type=query_variant`) из выбранной коллекции
2. Выполняет batched retrieval в Qdrant (`query_batch_points`)
3. Использует fusion в Qdrant (`Fusion.RRF`) для нескольких query-вариантов
4. Считает метрики по каждому эксперименту: `hit@k`, `mrr@k`, `recall_support_rows@k`
5. Запускает suite по сетке параметров и сохраняет summary/detail CSV

In [1]:
# Установите при первом запуске
%pip install -q qdrant-client pandas numpy tqdm

Note: you may need to restart the kernel to use updated packages.


## Конфигурация и базовые функции

Этот блок содержит:
- параметры эксперимента
- dataclass-конфиг
- утилиты нормализации и фильтрации уже завершенных запусков

In [18]:
import os
import re
import time
import itertools
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from qdrant_client import QdrantClient, models
from tqdm.auto import tqdm

# ----------------------
# Run config
# ----------------------

INDEXING_CHOICES = ['sparse_only', 'dense_only', 'hybrid']
RETRIEVAL_CHOICES = ['rrf', 'two_stage']
SERIALIZATION_CHOICES = ['markdown', 'json', 'sentence']
CHUNKING_CHOICES = ['row_level', 'schema_cell', 'table_level']
QUERY_EXPANSION_CHOICES = ['none', 'llm_generated']

_INT_RE = re.compile(r'^[+-]?\d+$')
_FLOAT_RE = re.compile(r'^[+-]?(?:\d+\.\d*|\.\d+|\d+)(?:[eE][+-]?\d+)?$')


@dataclass
class EvalConfig:
    indexing: str = 'hybrid'
    retrieval_strategy: str = 'rrf'
    serialization: str = 'markdown'
    chunking: str = 'row_level'
    query_expansion: str = 'none'
    encoding_budget: int = 512
    top_k: int = 10
    rerank_k: int = 50


def _normalize_indexing(indexing: str) -> str:
    m = {
        'dense_only': 'dense',
        'sparse_only': 'sparse',
        'hybrid': 'hybrid',
        'dense': 'dense',
        'sparse': 'sparse',
    }
    if indexing not in m:
        raise ValueError(f'Unknown indexing mode: {indexing}')
    return m[indexing]


def get_vector_names_for_cfg(cfg: EvalConfig) -> Tuple[List[str], List[str]]:
    idx = _normalize_indexing(cfg.indexing)

    if idx == 'dense':
        return (
            [f"dense_{cfg.serialization}_{cfg.chunking}_{cfg.encoding_budget}"],
            [f"dense_query_{cfg.query_expansion}"],
        )

    if idx == 'sparse':
        return (
            [f"sparse_{cfg.serialization}_{cfg.chunking}_{cfg.encoding_budget}"],
            [f"sparse_query_{cfg.query_expansion}"],
        )

    # hybrid: объединяем sparse + dense в одном fusion запросе Qdrant
    return (
        [
            f"dense_{cfg.serialization}_{cfg.chunking}_{cfg.encoding_budget}",
            f"sparse_{cfg.serialization}_{cfg.chunking}_{cfg.encoding_budget}",
        ],
        [
            f"dense_query_{cfg.query_expansion}",
            f"sparse_query_{cfg.query_expansion}",
        ],
    )


def make_grid(param_grid: Dict[str, List[Any]]) -> List[EvalConfig]:
    keys = sorted(param_grid.keys())
    combos = []
    for values in itertools.product(*(param_grid[k] for k in keys)):
        combos.append(EvalConfig(**dict(zip(keys, values))))
    return combos


def _normalize_cfg_value(v: Any) -> Any:
    if pd.isna(v):
        return None

    if isinstance(v, str):
        s = v.strip()
        sl = s.lower()
        if sl == 'true':
            return True
        if sl == 'false':
            return False
        if _INT_RE.fullmatch(s):
            return int(s)
        if _FLOAT_RE.fullmatch(s):
            fv = float(s)
            return int(fv) if fv.is_integer() else fv
        return s

    if isinstance(v, (np.integer, int)):
        return int(v)

    if isinstance(v, (np.floating, float)):
        fv = float(v)
        return int(fv) if fv.is_integer() else fv

    return v


def _cfg_key(cfg: EvalConfig) -> tuple:
    fields = list(EvalConfig.__dataclass_fields__.keys())
    return tuple(_normalize_cfg_value(getattr(cfg, f)) for f in fields)


def _row_key(row: pd.Series) -> tuple:
    fields = list(EvalConfig.__dataclass_fields__.keys())
    return tuple(_normalize_cfg_value(row.get(f)) for f in fields)


def exclude_completed_from_grid(
    grid: List[EvalConfig],
    results_csv_path: str,
    include_error_as_completed: bool = True,
    stage: str = 'retrieval_qdrant',
) -> List[EvalConfig]:
    p = Path(results_csv_path)
    if not p.exists():
        print(f'[WARN] Results file not found: {p}. Returning original grid ({len(grid)} configs).')
        return grid

    df = pd.read_csv(p)
    if df.empty:
        print('[INFO] Results CSV is empty. Nothing to exclude.')
        return grid

    if 'stage' in df.columns:
        df = df[df['stage'] == stage]

    if 'status' in df.columns:
        if include_error_as_completed:
            df = df[df['status'].isin(['ok', 'error'])]
        else:
            df = df[df['status'] == 'ok']

    completed_keys = set(_row_key(r) for _, r in df.iterrows())
    remaining = [cfg for cfg in grid if _cfg_key(cfg) not in completed_keys]
    skipped = len(grid) - len(remaining)
    print(
        f'Loaded {len(df)} completed rows from {p}. '
        f'Removed {skipped} configs from grid. Remaining: {len(remaining)}'
    )
    return remaining

## Подключение к Qdrant

Отдельный блок только для клиентского подключения и runtime-параметров окружения.

In [6]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# ----------------------
# Qdrant connection config
# ----------------------
QDRANT_HOST = os.getenv('QDRANT_HOST')
QDRANT_PORT = int(os.getenv('QDRANT_PORT', '80'))
QDRANT_API_KEY = os.getenv('QDRANT_API_KEY')
QDRANT_HTTPS = os.getenv('QDRANT_HTTPS', 'false').lower() == 'true'

# slugify(dense_model + '_' + dataset) в ingestion notebook
COLLECTION_NAME = os.getenv('QDRANT_COLLECTION')

QUERY_LIMIT = int(os.getenv('QUERY_LIMIT', '0'))  # 0 = all
SCROLL_BATCH_SIZE = int(os.getenv('SCROLL_BATCH_SIZE', '1000'))
QUERY_BATCH_REQUESTS = int(os.getenv('QUERY_BATCH_REQUESTS', '1000'))
REQUEST_TIMEOUT_SEC = int(os.getenv('QDRANT_TIMEOUT_SEC', '3000'))

OUT_DIR = Path('review_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

client = QdrantClient(
    host=QDRANT_HOST,
    port=QDRANT_PORT,
    api_key=QDRANT_API_KEY if QDRANT_API_KEY != 'CHANGE_ME' else None,
    https=QDRANT_HTTPS,
    timeout=REQUEST_TIMEOUT_SEC,
    prefer_grpc=True
)

print('Qdrant client initialized')
print('Collection:', COLLECTION_NAME)

## Проверка коллекции и доступных векторов

In [21]:
client.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='Qwen_Qwen3-Embedding-4B_Qdrant_bm25_custom'), CollectionDescription(name='Qwen_Qwen3-Embedding-4B_Qdrant_bm25_fetaqa'), CollectionDescription(name='Qwen_Qwen3-Embedding-4B_Qdrant_bm25_lighteval_wikitablequestions'), CollectionDescription(name='Qwen_Qwen3-Embedding-4B_Qdrant_bm25_open_wikitable'), CollectionDescription(name='intfloat_multilingual-e5-large-instruct_custom_baseline_recursive_dense_markdown'), CollectionDescription(name='intfloat_multilingual-e5-large-instruct_Qdrant_bm25_custom'), CollectionDescription(name='intfloat_multilingual-e5-large-instruct_naver_neuclir22-splade-ru_custom')])

In [22]:
# Посмотреть доступные vector names в коллекции
info = client.get_collection(COLLECTION_NAME)

dense_vectors = []
sparse_vectors = []

vectors_cfg = getattr(info.config.params, 'vectors', None)
if isinstance(vectors_cfg, dict):
    dense_vectors = list(vectors_cfg.keys())
elif vectors_cfg is not None and hasattr(vectors_cfg, 'size'):
    dense_vectors = ['default']

sparse_cfg = getattr(info.config.params, 'sparse_vectors', None)
if isinstance(sparse_cfg, dict):
    sparse_vectors = list(sparse_cfg.keys())

print('Dense vectors:', dense_vectors)
print('Sparse vectors:', sparse_vectors)

Dense vectors: ['dense_markdown_table_level_512', 'dense_query_none', 'dense_markdown_row_level_512']
Sparse vectors: ['sparse_markdown_table_level_512', 'sparse_query_none', 'sparse_markdown_row_level_512']


## Модели запросов и утилиты препроцессинга

In [23]:
@dataclass
class QueryRecord:
    query_id: str  # point.id из Qdrant — уникально для каждого query-варианта
    qid: str       # оригинальный qid для GT-matching
    question_original: str
    table_id: str
    supporting_rows: List[int]
    vectors: List[Any]


@dataclass
class QueryFusionRecord:
    query_id: str  # point.id из Qdrant — уникально для каждого query-варианта
    qid: str       # оригинальный qid для GT-matching
    question_original: str
    table_id: str
    supporting_rows: List[int]
    prefetch_specs: List[Tuple[str, Any]]  # (corpus_vector_name, query_vector)


def _to_int_or_none(v: Any) -> Optional[int]:
    if isinstance(v, bool):
        return None

    if isinstance(v, (int, np.integer)):
        return int(v)

    if isinstance(v, (float, np.floating)):
        fv = float(v)
        if np.isfinite(fv) and fv.is_integer():
            return int(fv)
        return None

    if isinstance(v, str):
        s = v.strip()
        if _INT_RE.fullmatch(s):
            return int(s)

    return None


def to_int_list(x: Any) -> List[int]:
    if not isinstance(x, list):
        return []

    out: List[int] = []
    for v in x:
        iv = _to_int_or_none(v)
        if iv is not None:
            out.append(iv)
    return out


def get_payload_value(payload: Optional[Dict[str, Any]], key: str, default: Any = None) -> Any:
    if not isinstance(payload, dict):
        return default
    return payload.get(key, default)


def extract_named_vector(point: Any, vector_name: str) -> Any:
    vecs = getattr(point, 'vector', None)
    if vecs is None:
        return None
    if isinstance(vecs, dict):
        return vecs.get(vector_name)
    return vecs


def to_sparse_vector_obj(v: Any) -> models.SparseVector:
    if isinstance(v, models.SparseVector):
        return v
    if hasattr(v, 'indices') and hasattr(v, 'values'):
        return models.SparseVector(indices=list(v.indices), values=list(v.values))
    if isinstance(v, dict) and 'indices' in v and 'values' in v:
        return models.SparseVector(indices=list(v['indices']), values=list(v['values']))
    raise ValueError(f'Cannot convert sparse vector from type: {type(v)}')


def is_sparse_vector_name(name: str) -> bool:
    return str(name).startswith('sparse_')


def query_filter_query_variants(query_expansion: str) -> models.Filter:
    return models.Filter(
        must=[
            models.FieldCondition(key='record_type', match=models.MatchValue(value='query_variant')),
            models.FieldCondition(key='query_expansion', match=models.MatchValue(value=query_expansion)),
        ]
    )


def query_filter_corpus_only() -> models.Filter:
    return models.Filter(
        must_not=[
            models.FieldCondition(key='record_type', match=models.MatchValue(value='query_variant')),
        ]
    )


def scroll_all_points(
    collection_name: str,
    scroll_filter: Optional[models.Filter],
    with_payload: bool,
    with_vectors: Any,
    batch_size: int = 512,
) -> List[Any]:
    all_points = []
    offset = None

    while True:
        resp = client.scroll(
            collection_name=collection_name,
            scroll_filter=scroll_filter,
            with_payload=with_payload,
            with_vectors=with_vectors,
            limit=batch_size,
            offset=offset,
        )

        if isinstance(resp, tuple):
            points, next_offset = resp
        else:
            points = getattr(resp, 'points', [])
            next_offset = getattr(resp, 'next_page_offset', None)

        if not points:
            break

        all_points.extend(points)
        offset = next_offset
        if offset is None:
            break

    return all_points


def _build_prefetch(query_vector: Any, vector_name: str, top_k: int) -> models.Prefetch:
    # Для Prefetch в этой версии клиента query должен быть raw-вектором,
    # а имя векторного пространства задается через using.
    if is_sparse_vector_name(vector_name):
        return models.Prefetch(
            query=to_sparse_vector_obj(query_vector),
            using=vector_name,
            limit=top_k,
            filter=models.Filter(
                must=[
                    models.HasVectorCondition(has_vector=vector_name),
                ],
            )
        )

    return models.Prefetch(
        query=list(query_vector),
        using=vector_name,
        limit=top_k,
        filter=models.Filter(
            must=[
                models.HasVectorCondition(has_vector=vector_name),
            ],
        )
    )


def _extract_points_from_batch_result(batch_result: Any) -> List[Any]:
    if isinstance(batch_result, list):
        return batch_result
    if hasattr(batch_result, 'points'):
        return batch_result.points
    if hasattr(batch_result, 'result') and hasattr(batch_result.result, 'points'):
        return batch_result.result.points
    return []


def _is_payload_too_large_error(e: Exception) -> bool:
    msg = str(e).lower()
    return (
        'payload error' in msg
        or 'larger than allowed' in msg
        or '413' in msg

        or 'request entity too large' in msg    )

## Метрики retrieval

Здесь только функции оценки качества поиска (релевантность, hit@k, mrr@k, recall).

In [24]:
def is_relevant(point_payload: Dict[str, Any], table_id: str, supporting_rows: List[int]) -> bool:
    if get_payload_value(point_payload, 'table_id') != table_id:
        return False

    if not supporting_rows:
        return True

    row_ids = set(to_int_list(get_payload_value(point_payload, 'row_ids', [])))
    return len(row_ids.intersection(set(supporting_rows))) > 0


def hit_at_k(results: List[Any], table_id: str, supporting_rows: List[int], k: int) -> float:
    for p in results[:k]:
        payload = getattr(p, 'payload', {}) or {}
        if is_relevant(payload, table_id, supporting_rows):
            return 1.0
    return 0.0


def mrr_at_k(results: List[Any], table_id: str, supporting_rows: List[int], k: int) -> float:
    for i, p in enumerate(results[:k], start=1):
        payload = getattr(p, 'payload', {}) or {}
        if is_relevant(payload, table_id, supporting_rows):
            return 1.0 / i
    return 0.0


def recall_support_rows_at_k(results: List[Any], table_id: str, supporting_rows: List[int], k: int) -> float:
    if not supporting_rows:
        return hit_at_k(results, table_id, supporting_rows, k)

    target = set(supporting_rows)
    covered = set()

    for p in results[:k]:
        payload = getattr(p, 'payload', {}) or {}
        if get_payload_value(payload, 'table_id') != table_id:
            continue
        row_ids = set(to_int_list(get_payload_value(payload, 'row_ids', [])))
        covered.update(row_ids.intersection(target))

    return len(covered) / len(target) if target else 0.0

## Retrieval движок (RRF и Two-Stage)

In [25]:
# Retrieval runners with progress, adaptive batching and two-stage strategy

def _merge_filter_with_has_id(
    base_filter: Optional[models.Filter],
    point_ids: List[Any],
) -> models.Filter:
    must = []
    must_not = []
    should = []

    if base_filter is not None:
        must.extend(list(getattr(base_filter, 'must', []) or []))
        must_not.extend(list(getattr(base_filter, 'must_not', []) or []))
        should.extend(list(getattr(base_filter, 'should', []) or []))

    must.append(models.HasIdCondition(has_id=point_ids))
    return models.Filter(
        must=must if must else None,
        must_not=must_not if must_not else None,
        should=should if should else None,
    )


def _make_rrf_query_request(
    prefetches: List[models.Prefetch],
    limit: int,
    query_filter: Optional[models.Filter],
    with_payload: bool,
) -> models.QueryRequest:
    return models.QueryRequest(
        prefetch=prefetches,
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        filter=query_filter,
        limit=limit,
        with_payload=with_payload,
        with_vector=False,
    )


def _build_prefetches(prefetch_specs: List[Tuple[str, Any]], top_k: int) -> List[models.Prefetch]:
    return [_build_prefetch(v, vn, top_k) for vn, v in prefetch_specs]


def _split_prefetch_specs(prefetch_specs: List[Tuple[str, Any]]) -> Tuple[List[Tuple[str, Any]], List[Tuple[str, Any]]]:
    sparse_specs: List[Tuple[str, Any]] = []
    dense_specs: List[Tuple[str, Any]] = []
    for vn, vec in prefetch_specs:
        if is_sparse_vector_name(vn):
            sparse_specs.append((vn, vec))
        else:
            dense_specs.append((vn, vec))
    return sparse_specs, dense_specs


def _iter_chunks(items: List[Any], chunk_size: int) -> List[List[Any]]:
    chunk_size = max(1, int(chunk_size))
    return [items[i:i + chunk_size] for i in range(0, len(items), chunk_size)]


def _query_batch_points_safe(
    collection_name: str,
    reqs: List[models.QueryRequest],
) -> Tuple[Optional[List[List[Any]]], Optional[Exception]]:
    try:
        resp = client.query_batch_points(
            collection_name=collection_name,
            requests=reqs,
            timeout=REQUEST_TIMEOUT_SEC,
        )
    except Exception as e:
        return None, e

    if isinstance(resp, list):
        return [_extract_points_from_batch_result(r) for r in resp], None

    results = getattr(resp, 'result', None)
    if isinstance(results, list):
        return [_extract_points_from_batch_result(r) for r in results], None

    return [[] for _ in reqs], None


def _query_point_safe(collection_name: str, req: models.QueryRequest) -> List[Any]:
    try:
        r = client.query_points(
            collection_name=collection_name,
            prefetch=req.prefetch,
            query=req.query,
            query_filter=req.filter,
            limit=req.limit,
            with_payload=req.with_payload,
            with_vectors=False,
        )
    except Exception:
        return []

    return _extract_points_from_batch_result(r)


def _execute_query_request_rows(
    collection_name: str,
    request_rows: List[Tuple[str, models.QueryRequest]],
    request_batch_size: int = QUERY_BATCH_REQUESTS,
    progress_desc: str = 'Qdrant processed query_ids',
) -> Dict[str, List[Any]]:
    out: Dict[str, List[Any]] = {query_id: [] for query_id, _ in request_rows}
    if not request_rows:
        print(f'{progress_desc}: 0/0 (non-empty results: 0)')
        return out

    i = 0
    current_batch = max(1, int(request_batch_size))
    total = len(request_rows)

    with tqdm(total=total, desc=progress_desc) as pbar:
        while i < total:
            size = min(current_batch, total - i)
            rows = request_rows[i:i + size]
            query_ids = [x[0] for x in rows]
            reqs = [x[1] for x in rows]

            batch_results, error = _query_batch_points_safe(collection_name, reqs)
            if error is None and batch_results is not None:
                for query_id, points in zip(query_ids, batch_results):
                    out[query_id] = points
                i += size
                pbar.update(size)
                continue

            if error is not None and _is_payload_too_large_error(error) and size > 1:
                current_batch = max(1, size // 2)
                print(f'[WARN] Payload too large, reduce query_batch_points batch to {current_batch}')
                continue

            print(f'[WARN] query_batch_points chunk failed, fallback per-query for {size} requests: {error}')
            for query_id, req in rows:
                out[query_id] = _query_point_safe(collection_name, req)

            i += size
            pbar.update(size)

    non_empty = sum(1 for v in out.values() if v)
    print(f'{progress_desc}: {i}/{total} (non-empty results: {non_empty})')
    return out


def batch_search_with_qdrant_fusion(
    collection_name: str,
    grouped_queries: List[QueryFusionRecord],
    top_k: int,
    corpus_filter: Optional[models.Filter],
    request_batch_size: int = QUERY_BATCH_REQUESTS,
) -> Dict[str, List[Any]]:
    request_rows: List[Tuple[str, models.QueryRequest]] = []
    for q in grouped_queries:
        prefetches = _build_prefetches(q.prefetch_specs, top_k)
        if not prefetches:
            continue
        req = _make_rrf_query_request(
            prefetches=prefetches,
            limit=top_k,
            query_filter=corpus_filter,
            with_payload=True,
        )
        request_rows.append((q.query_id, req))

    return _execute_query_request_rows(
        collection_name=collection_name,
        request_rows=request_rows,
        request_batch_size=request_batch_size,
        progress_desc='Qdrant processed query_ids',
    )


def batch_search_with_qdrant_two_stage(
    collection_name: str,
    grouped_queries: List[QueryFusionRecord],
    top_k: int,
    rerank_k: int,
    corpus_filter: Optional[models.Filter],
    request_batch_size: int = QUERY_BATCH_REQUESTS,
    query_chunk_size: int = 100,
) -> Dict[str, List[Any]]:
    """
    Two-stage retrieval полностью на Qdrant, но без удержания всего stage1_results в памяти.

    Обрабатываем запросы чанками:
    1) sparse candidate generation для чанка
    2) сразу строим stage2 dense rerank для этого же чанка
    3) сохраняем только финальные результаты
    """
    out: Dict[str, List[Any]] = {}
    stage1_limit = max(top_k, rerank_k)

    for chunk in tqdm(_iter_chunks(grouped_queries, query_chunk_size), desc='Two-stage query chunks'):
        stage1_rows: List[Tuple[str, models.QueryRequest]] = []
        stage2_meta: Dict[str, List[Tuple[str, Any]]] = {}
        fallback_rows: List[Tuple[str, models.QueryRequest]] = []

        for q in chunk:
            sparse_specs, dense_specs = _split_prefetch_specs(q.prefetch_specs)

            if not sparse_specs or not dense_specs:
                prefetches = _build_prefetches(q.prefetch_specs, top_k)
                if prefetches:
                    fallback_rows.append((
                        q.query_id,
                        _make_rrf_query_request(
                            prefetches=prefetches,
                            limit=top_k,
                            query_filter=corpus_filter,
                            with_payload=True,
                        ),
                    ))
                continue

            sparse_prefetch = _build_prefetches(sparse_specs, stage1_limit)
            stage1_rows.append((
                q.query_id,
                _make_rrf_query_request(
                    prefetches=sparse_prefetch,
                    limit=stage1_limit,
                    query_filter=corpus_filter,
                    with_payload=False,
                ),
            ))
            stage2_meta[q.query_id] = dense_specs

        stage1_results = _execute_query_request_rows(
            collection_name=collection_name,
            request_rows=stage1_rows,
            request_batch_size=request_batch_size,
            progress_desc='Qdrant stage1 sparse query_ids',
        )

        stage2_rows: List[Tuple[str, models.QueryRequest]] = []
        for query_id, dense_specs in stage2_meta.items():
            cands = stage1_results.get(query_id, [])
            cand_ids = [getattr(p, 'id', None) for p in cands if getattr(p, 'id', None) is not None]
            if not cand_ids:
                continue

            dense_prefetch = _build_prefetches(dense_specs, top_k)
            q_filter = _merge_filter_with_has_id(corpus_filter, cand_ids)
            stage2_rows.append((
                query_id,
                _make_rrf_query_request(
                    prefetches=dense_prefetch,
                    limit=top_k,
                    query_filter=q_filter,
                    with_payload=True,
                ),
            ))

        stage2_results = _execute_query_request_rows(
            collection_name=collection_name,
            request_rows=stage2_rows,
            request_batch_size=request_batch_size,
            progress_desc='Qdrant stage2 dense query_ids',
        )

        if fallback_rows:
            fallback_results = _execute_query_request_rows(
                collection_name=collection_name,
                request_rows=fallback_rows,
                request_batch_size=request_batch_size,
                progress_desc='Qdrant fallback fusion query_ids',
            )
            out.update(fallback_results)

        out.update(stage2_results)

        del stage1_results
        del stage2_rows
        del stage1_rows
        del stage2_meta
        del fallback_rows

    return out

## Загрузка query-векторов из Qdrant

In [26]:
def get_collection_vector_names(collection_name: str) -> Tuple[List[str], List[str]]:
    info = client.get_collection(collection_name)

    dense_vectors: List[str] = []
    sparse_vectors: List[str] = []

    vectors_cfg = getattr(info.config.params, 'vectors', None)
    if isinstance(vectors_cfg, dict):
        dense_vectors = list(vectors_cfg.keys())
    elif vectors_cfg is not None and hasattr(vectors_cfg, 'size'):
        dense_vectors = ['default']

    sparse_cfg = getattr(info.config.params, 'sparse_vectors', None)
    if isinstance(sparse_cfg, dict):
        sparse_vectors = list(sparse_cfg.keys())

    return dense_vectors, sparse_vectors


def load_grouped_queries(
    collection_name: str,
    query_vector_name: str,
    query_expansion: str,
    query_limit: int = 0,
) -> List[QueryRecord]:
    q_filter = query_filter_query_variants(query_expansion)
    points = scroll_all_points(
        collection_name=collection_name,
        scroll_filter=q_filter,
        with_payload=True,
        with_vectors=[query_vector_name],
        batch_size=SCROLL_BATCH_SIZE,
    )

    # Теперь группируем по point.id (уникально для каждого query-варианта)
    grouped: Dict[str, QueryRecord] = {}
    for p in points:
        payload = getattr(p, 'payload', {}) or {}
        point_id = str(getattr(p, 'id', ''))
        qid = str(get_payload_value(payload, 'qid', ''))
        
        if not point_id or not qid:
            continue

        vec = extract_named_vector(p, query_vector_name)
        if vec is None:
            continue

        # Создаём одну запись per point.id (каждый query-вариант отдельно)
        if point_id not in grouped:
            grouped[point_id] = QueryRecord(
                query_id=point_id,
                qid=qid,
                question_original=str(get_payload_value(payload, 'question_original', '')),
                table_id=str(get_payload_value(payload, 'table_id', '')),
                supporting_rows=to_int_list(get_payload_value(payload, 'supporting_rows', [])),
                vectors=[],
            )

        grouped[point_id].vectors.append(vec)

    queries = [q for q in grouped.values() if q.table_id and q.vectors]
    if query_limit > 0:
        queries = queries[:query_limit]

    return queries


dense_names, sparse_names = get_collection_vector_names(COLLECTION_NAME)
print('Dense vectors:', dense_names)
print('Sparse vectors:', sparse_names)

Dense vectors: ['dense_markdown_table_level_512', 'dense_query_none', 'dense_markdown_row_level_512']
Sparse vectors: ['sparse_markdown_table_level_512', 'sparse_query_none', 'sparse_markdown_row_level_512']


## Оркестрация экспериментов (функции)

In [27]:
def run_one_experiment(
    cfg: EvalConfig,
    collection_name: str,
    queries_cache: Dict[Tuple[str, str], List[QueryRecord]],
) -> Tuple[Dict[str, Any], pd.DataFrame]:
    t0 = time.time()

    corpus_vector_names, query_vector_names = get_vector_names_for_cfg(cfg)
    if len(corpus_vector_names) != len(query_vector_names):
        raise ValueError('Corpus/query vector name lists length mismatch')

    corpus_filter = query_filter_corpus_only()

    query_sets = []
    for qv_name in query_vector_names:
        cache_key = (qv_name, cfg.query_expansion)
        if cache_key not in queries_cache:
            queries_cache[cache_key] = load_grouped_queries(
                collection_name=collection_name,
                query_vector_name=qv_name,
                query_expansion=cfg.query_expansion,
                query_limit=QUERY_LIMIT,
            )
        query_sets.append((qv_name, queries_cache[cache_key]))

    # Базовый набор query_id берём из первого query-vector space
    base_map: Dict[str, QueryFusionRecord] = {
        q.query_id: QueryFusionRecord(
            query_id=q.query_id,
            qid=q.qid,
            question_original=q.question_original,
            table_id=q.table_id,
            supporting_rows=q.supporting_rows,
            prefetch_specs=[],
        )
        for q in (query_sets[0][1] if query_sets else [])
    }

    # Добавляем prefetch из каждого векторного пространства (dense/sparse)
    for corpus_vn, (_, qrecords) in zip(corpus_vector_names, query_sets):
        q_vectors_by_query_id = {q.query_id: q.vectors for q in qrecords}
        for query_id, fusion_q in base_map.items():
            for vec in q_vectors_by_query_id.get(query_id, []):
                fusion_q.prefetch_specs.append((corpus_vn, vec))

    grouped_queries = [q for q in base_map.values() if q.prefetch_specs]

    idx = _normalize_indexing(cfg.indexing)
    use_two_stage = cfg.retrieval_strategy == 'two_stage' and idx == 'hybrid'

    if cfg.retrieval_strategy == 'two_stage' and not use_two_stage:
        print(f'[WARN] two_stage currently supports hybrid only. Fallback to rrf for indexing={cfg.indexing}')

    if use_two_stage:
        results_by_query_id = batch_search_with_qdrant_two_stage(
            collection_name=collection_name,
            grouped_queries=grouped_queries,
            top_k=cfg.top_k,
            rerank_k=cfg.rerank_k,
            corpus_filter=corpus_filter,
        )
    else:
        results_by_query_id = batch_search_with_qdrant_fusion(
            collection_name=collection_name,
            grouped_queries=grouped_queries,
            top_k=cfg.top_k,
            corpus_filter=corpus_filter,
        )

    rows = []
    for q in tqdm(
        grouped_queries,
        desc=f"Questions ({cfg.indexing}, {cfg.query_expansion}, {cfg.retrieval_strategy})",
    ):
        merged = results_by_query_id.get(q.query_id, [])
        hit = hit_at_k(merged, q.table_id, q.supporting_rows, cfg.top_k)
        mrr = mrr_at_k(merged, q.table_id, q.supporting_rows, cfg.top_k)
        recall_rows = recall_support_rows_at_k(merged, q.table_id, q.supporting_rows, cfg.top_k)

        top_ids = [str(getattr(p, 'id', '')) for p in merged]
        top_tables = [str(get_payload_value(getattr(p, 'payload', {}) or {}, 'table_id', '')) for p in merged]

        prefetch_using_names = list(dict.fromkeys(vn for vn, _ in q.prefetch_specs))

        rows.append({
            'query_id': q.query_id,
            'qid': q.qid,
            'question_original': q.question_original,
            'table_id': q.table_id,
            'supporting_rows': q.supporting_rows,
            'n_prefetch_vectors': len(q.prefetch_specs),
            'prefetch_using_vector_names': prefetch_using_names,
            'prefetch_using_vector_names_str': '|'.join(prefetch_using_names),
            'hit@k': hit,
            'mrr@k': mrr,
            'recall_support_rows@k': recall_rows,
            'top_point_ids': top_ids,
            'top_table_ids': top_tables,
        })

    df_detail = pd.DataFrame(rows)

    summary = {
        **asdict(cfg),
        'stage': 'retrieval_qdrant',
        'collection': collection_name,
        'corpus_vector_names': '|'.join(corpus_vector_names),
        'query_vector_names': '|'.join(query_vector_names),
        'n_queries': int(len(df_detail)),
        'hit@k': float(df_detail['hit@k'].mean()) if len(df_detail) else 0.0,
        'mrr@k': float(df_detail['mrr@k'].mean()) if len(df_detail) else 0.0,
        'recall_support_rows@k': float(df_detail['recall_support_rows@k'].mean()) if len(df_detail) else 0.0,
        'runtime_sec': round(time.time() - t0, 3),
        'status': 'ok',
    }
    return summary, df_detail


def run_experiment_suite(
    configs: List[EvalConfig],
    collection_name: str,
    out_dir: str = 'review_outputs',
    tag: str = 'qdrant_eval',
) -> Tuple[pd.DataFrame, pd.DataFrame, Path, Path]:
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    ts = time.strftime('%Y%m%d_%H%M%S')
    summary_path = out / f'{tag}_summary_{ts}.csv'
    detail_path = out / f'{tag}_detail_{ts}.csv'

    dense_names, sparse_names = get_collection_vector_names(collection_name)
    available = set(dense_names) | set(sparse_names)

    summary_rows: List[Dict[str, Any]] = []
    detail_rows: List[Dict[str, Any]] = []
    queries_cache: Dict[Tuple[str, str], List[QueryRecord]] = {}

    for cfg in tqdm(configs, desc='Qdrant experiment grid'):
        corpus_vector_names, query_vector_names = get_vector_names_for_cfg(cfg)
        missing = [v for v in (corpus_vector_names + query_vector_names) if v not in available]

        if missing:
            summary_rows.append({
                **asdict(cfg),
                'stage': 'retrieval_qdrant',
                'collection': collection_name,
                'corpus_vector_names': '|'.join(corpus_vector_names),
                'query_vector_names': '|'.join(query_vector_names),
                'status': 'error',
                'error': f'Vector name not found in collection: {missing}',
                'runtime_sec': 0.0,
            })
            continue

        try:
            summary, df_detail = run_one_experiment(
                cfg=cfg,
                collection_name=collection_name,
                queries_cache=queries_cache,
            )
            summary_rows.append(summary)

            if not df_detail.empty:
                df_detail = df_detail.copy()
                for k, v in asdict(cfg).items():
                    df_detail[k] = v
                df_detail['collection'] = collection_name
                df_detail['corpus_vector_names'] = '|'.join(corpus_vector_names)
                df_detail['query_vector_names'] = '|'.join(query_vector_names)
                detail_rows.extend(df_detail.to_dict(orient='records'))
        except Exception as e:
            summary_rows.append({
                **asdict(cfg),
                'stage': 'retrieval_qdrant',
                'collection': collection_name,
                'corpus_vector_names': '|'.join(corpus_vector_names),
                'query_vector_names': '|'.join(query_vector_names),
                'status': 'error',
                'error': str(e),
                'runtime_sec': 0.0,
            })

        pd.DataFrame(summary_rows).to_csv(summary_path, index=False)
        if detail_rows:
            pd.DataFrame(detail_rows).to_csv(detail_path, index=False)

    df_summary = pd.DataFrame(summary_rows)
    df_detail_all = pd.DataFrame(detail_rows)

    if not df_detail_all.empty:
        df_detail_all.to_csv(detail_path, index=False)
    df_summary.to_csv(summary_path, index=False)

    return df_summary, df_detail_all, summary_path, detail_path

### Построение grid конфигураций

In [28]:
def make_grid_from_config(config: Dict[str, Any]) -> List[EvalConfig]:
    """
    Генерация сетки с поддержкой условных параметров.

    Формат конфига:
    1) Обычный (как раньше):
       {'indexing': [...], 'retrieval_strategy': [...], ...}
    2) Условный параметр:
       {
         'indexing': ['sparse_only', 'dense_only', 'hybrid'],
         'retrieval_strategy': {
             'depends_on': 'indexing',
             'values': {
                 'hybrid': ['rrf', 'two_stage'],
                 '*': ['rrf']
             }
         },
         ...
       }
    """
    conditional_specs: Dict[str, Dict[str, Any]] = {}
    base_params: Dict[str, List[Any]] = {}

    for key, val in config.items():
        if isinstance(val, dict) and 'depends_on' in val and 'values' in val:
            conditional_specs[key] = val
            continue

        if not isinstance(val, list):
            raise ValueError(f'Grid param `{key}` must be list or conditional spec dict')
        base_params[key] = val

    keys = sorted(base_params.keys())
    if not keys:
        return []

    out: List[EvalConfig] = []
    for values in itertools.product(*(base_params[k] for k in keys)):
        row = dict(zip(keys, values))
        rows = [row]

        for field, spec in conditional_specs.items():
            dep_key = str(spec['depends_on'])
            mapping = spec['values'] or {}
            dep_val = row.get(dep_key)

            allowed = mapping.get(dep_val, mapping.get('*', []))
            if not isinstance(allowed, list):
                allowed = [allowed]

            next_rows = []
            for r in rows:
                for a in allowed:
                    nr = dict(r)
                    nr[field] = a
                    next_rows.append(nr)
            rows = next_rows

        for r in rows:
            out.append(EvalConfig(**r))

    return out

## Запуск experiment suite

In [29]:
stage1_qdrant_config = {
    'indexing': ['sparse_only', 'dense_only', 'hybrid'],
    'retrieval_strategy': {
        'depends_on': 'indexing',
        'values': {
            'hybrid': ['rrf', 'two_stage'],
            '*': ['rrf'],
        },
    },
    'serialization': ['markdown', ], # 'json', 'sentence'
    'chunking': ['row_level', 'table_level', ], #  'schema_cell'
    'query_expansion': ['none', ], # 'llm_generated'
    'encoding_budget': [512],
    'top_k': [5, 10],
    'rerank_k': [50],
}
stage1_qdrant_grid = make_grid_from_config(stage1_qdrant_config)

In [ ]:

# Optional: исключить уже прогнанные конфиги из предыдущих summary
RESULTS_SUMMARY_PATHS = os.getenv(
    'QDRANT_RESULTS_SUMMARY_PATHS',
    '',
    # 'review_outputs/qdrant_retrieval_summary_20260429_074422.csv,review_outputs/qdrant_retrieval_summary_20260429_090748.csv'
    # 'review_outputs/qdrant_retrieval_summary_20260428_073924.csv',
    # 'review_outputs/qdrant_retrieval_summary_20260420_203609.csv,review_outputs/qdrant_retrieval_summary_20260421_072026.csv',
    # 'review_outputs/qdrant_retrieval_summary_20260403_173311.csv', 
    # 'review_outputs/qdrant_retrieval_summary_20260329_080222.csv,review_outputs/qdrant_retrieval_summary_20260328_192829.csv',
    # 'review_outputs/qdrant_retrieval_summary_20260324_170131.csv',
    # 'review_outputs/qdrant_retrieval_summary_20260323_132631.csv,review_outputs/qdrant_retrieval_summary_20260323_162105.csv,review_outputs/qdrant_retrieval_summary_20260323_193942.csv'
)

# Поддерживаются разделители: запятая, ; и перенос строки
for ch in [';', '\n', '\t']:
    RESULTS_SUMMARY_PATHS = RESULTS_SUMMARY_PATHS.replace(ch, ',')
summary_paths = [p.strip() for p in RESULTS_SUMMARY_PATHS.split(',') if p.strip()]

if summary_paths:
    for p in summary_paths:
        stage1_qdrant_grid = exclude_completed_from_grid(
            stage1_qdrant_grid,
            p,
            include_error_as_completed=True,
            stage='retrieval_qdrant',
        )

df_summary, df_detail, summary_path, detail_path = run_experiment_suite(
    configs=stage1_qdrant_grid,
    collection_name=COLLECTION_NAME,
    out_dir=str(OUT_DIR),
    tag='qdrant_retrieval',
)

display(df_summary.head(20))
# Теперь показываем query_id и qid для отслеживания варианта
display(df_detail[['query_id', 'qid', 'prefetch_using_vector_names_str', 'hit@k', 'mrr@k']].head(20))
print('Summary file:', summary_path)
print('Detail file:', detail_path)

## Baseline retrieval

In [ ]:
# --- Baseline evaluation using existing retrieval/eval pipeline ---
BASELINE_COLLECTION_NAME = os.getenv(
    'QDRANT_BASELINE_COLLECTION',
)
BASELINE_CORPUS_VECTOR_NAME = os.getenv(
    'QDRANT_BASELINE_CORPUS_VECTOR',
    'dense_markdown_recursive_512',
)
BASELINE_QUERY_VECTOR_NAME = os.getenv(
    'QDRANT_BASELINE_QUERY_VECTOR',
    'dense_query_none',
)
BASELINE_QUERY_EXPANSION = os.getenv('QDRANT_BASELINE_QUERY_EXPANSION', 'none')
BASELINE_TOP_K = int(os.getenv('QDRANT_BASELINE_TOP_K', '10'))

dense_names_b, sparse_names_b = get_collection_vector_names(BASELINE_COLLECTION_NAME)
available_b = set(dense_names_b) | set(sparse_names_b)
required_b = {BASELINE_CORPUS_VECTOR_NAME, BASELINE_QUERY_VECTOR_NAME}
missing_b = sorted(required_b - available_b)
if missing_b:
    raise ValueError(
        f'Baseline vector name not found in collection {BASELINE_COLLECTION_NAME}: {missing_b}'
    )

baseline_queries = load_grouped_queries(
    collection_name=BASELINE_COLLECTION_NAME,
    query_vector_name=BASELINE_QUERY_VECTOR_NAME,
    query_expansion=BASELINE_QUERY_EXPANSION,
    query_limit=QUERY_LIMIT,
 )

baseline_grouped_queries = [
    QueryFusionRecord(
        query_id=q.query_id,
        qid=q.qid,
        question_original=q.question_original,
        table_id=q.table_id,
        supporting_rows=q.supporting_rows,
        prefetch_specs=[(BASELINE_CORPUS_VECTOR_NAME, vec) for vec in q.vectors],
    )
    for q in baseline_queries
    if q.table_id and q.vectors
 ]

baseline_results = batch_search_with_qdrant_fusion(
    collection_name=BASELINE_COLLECTION_NAME,
    grouped_queries=baseline_grouped_queries,
    top_k=BASELINE_TOP_K,
    corpus_filter=query_filter_corpus_only(),
    request_batch_size=QUERY_BATCH_REQUESTS,
 )

baseline_rows = []
for q in tqdm(baseline_grouped_queries, desc='Baseline questions'):
    merged = baseline_results.get(q.query_id, [])
    prefetch_using_names = list(dict.fromkeys(vn for vn, _ in q.prefetch_specs))
    baseline_rows.append({
        'query_id': q.query_id,
        'qid': q.qid,
        'question_original': q.question_original,
        'table_id': q.table_id,
        'supporting_rows': q.supporting_rows,
        'n_prefetch_vectors': len(q.prefetch_specs),
            'prefetch_using_vector_names': prefetch_using_names,
            'prefetch_using_vector_names_str': '|'.join(prefetch_using_names),
        'hit@k': hit_at_k(merged, q.table_id, q.supporting_rows, BASELINE_TOP_K),
        'mrr@k': mrr_at_k(merged, q.table_id, q.supporting_rows, BASELINE_TOP_K),
        'recall_support_rows@k': recall_support_rows_at_k(
            merged, q.table_id, q.supporting_rows, BASELINE_TOP_K
        ),
        'top_point_ids': [str(getattr(p, 'id', '')) for p in merged],
        'top_table_ids': [
            str(get_payload_value(getattr(p, 'payload', {}) or {}, 'table_id', ''))
            for p in merged
        ],
    })

baseline_detail = pd.DataFrame(baseline_rows)
baseline_summary = pd.DataFrame([
    {
        'stage': 'retrieval_qdrant_baseline',
        'collection': BASELINE_COLLECTION_NAME,
        'corpus_vector_names': BASELINE_CORPUS_VECTOR_NAME,
        'query_vector_names': BASELINE_QUERY_VECTOR_NAME,
        'query_expansion': BASELINE_QUERY_EXPANSION,
        'top_k': BASELINE_TOP_K,
        'n_queries': int(len(baseline_detail)),
        'hit@k': float(baseline_detail['hit@k'].mean()) if len(baseline_detail) else 0.0,
        'mrr@k': float(baseline_detail['mrr@k'].mean()) if len(baseline_detail) else 0.0,
        'recall_support_rows@k': float(baseline_detail['recall_support_rows@k'].mean()) if len(baseline_detail) else 0.0,
        'status': 'ok',
    }
])

display(baseline_summary)
display(
    baseline_detail[
        ['query_id', 'qid', 'hit@k', 'mrr@k', 'recall_support_rows@k']
    ].head(20)
)

ts = time.strftime('%Y%m%d_%H%M%S')
baseline_summary_path = OUT_DIR / f'qdrant_retrieval_baseline_summary_{ts}.csv'
baseline_detail_path = OUT_DIR / f'qdrant_retrieval_baseline_detail_{ts}.csv'
baseline_summary.to_csv(baseline_summary_path, index=False)
baseline_detail.to_csv(baseline_detail_path, index=False)
print('Baseline summary file:', baseline_summary_path)
print('Baseline detail file:', baseline_detail_path)